In [1]:
import pandas as pd
import numpy as np

In [2]:
raw_data = pd.read_csv("C:/Users/Durosimi/PROJECTS/Parkinson/Parkinsons-Telemonitoring-ucirvine.csv")

In [3]:
raw_data.head()

,subject,age,sex,test_time,motor_updrs,total_updrs,jitter,jitter_abs,jitter_rap,jitter_ppq5,...,shimmer_db,shimmer_apq3,shimmer_apq5,shimmer_apq11,shimmer_dda,nhr,hnr,rpde,dfa,ppe
0,1,72,False,5.6431,28.199,34.398,0.00662,0.000034,0.00401,0.00317,...,0.230,0.01438,0.01309,0.01662,0.04314,0.014290,21.640,0.41888,0.54842,0.16006
1,1,72,False,12.6660,28.447,34.894,0.00300,0.000017,0.00132,0.00150,...,0.179,0.00994,0.01072,0.01689,0.02982,0.011112,27.183,0.43493,0.56477,0.10810
2,1,72,False,19.6810,28.695,35.389,0.00481,0.000025,0.00205,0.00208,...,0.181,0.00734,0.00844,0.01458,0.02202,0.020220,23.047,0.46222,0.54405,0.21014
3,1,72,False,25.6470,28.905,35.810,0.00528,0.000027,0.00191,0.00264,...,0.327,0.01106,0.01265,0.01963,0.03317,0.027837,24.445,0.48730,0.57794,0.33277
4,1,72,False,33.6420,29.187,36.375,0.00335,0.000020,0.00093,0.00130,...,0.176,0.00679,0.00929,0.01819,0.02036,0.011625,26.126,0.47188,0.56122,0.19361


In [4]:
raw_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5875 entries, 0 to 5874
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   subject        5875 non-null   int64  
 1   age            5875 non-null   int64  
 2   sex            5875 non-null   bool   
 3   test_time      5875 non-null   float64
 4   motor_updrs    5875 non-null   float64
 5   total_updrs    5875 non-null   float64
 6   jitter         5875 non-null   float64
 7   jitter_abs     5875 non-null   float64
 8   jitter_rap     5875 non-null   float64
 9   jitter_ppq5    5875 non-null   float64
 10  jitter_ddp     5875 non-null   float64
 11  shimmer        5875 non-null   float64
 12  shimmer_db     5875 non-null   float64
 13  shimmer_apq3   5875 non-null   float64
 14  shimmer_apq5   5875 non-null   float64
 15  shimmer_apq11  5875 non-null   float64
 16  shimmer_dda    5875 non-null   float64
 17  nhr            5875 non-null   float64
 18  hnr     

In [5]:
raw_data.columns

Index(['subject', 'age', 'sex', 'test_time', 'motor_updrs', 'total_updrs',
       'jitter', 'jitter_abs', 'jitter_rap', 'jitter_ppq5', 'jitter_ddp',
       'shimmer', 'shimmer_db', 'shimmer_apq3', 'shimmer_apq5',
       'shimmer_apq11', 'shimmer_dda', 'nhr', 'hnr', 'rpde', 'dfa', 'ppe'],
      dtype='object')

In [6]:
id_column = ['subject']
time_column = ['test_time']
target = ['motor_updrs', 'total_updrs']

meta_cols = [id_column, time_column, target, "sex", "age"]

In [7]:
data = raw_data.copy()

In [8]:

if data["sex"].dtype == bool:
    data["sex"] = data["sex"].astype(int)

# Check missing values
data.isna().sum()


subject          0
age              0
sex              0
test_time        0
motor_updrs      0
total_updrs      0
jitter           0
jitter_abs       0
jitter_rap       0
jitter_ppq5      0
jitter_ddp       0
shimmer          0
shimmer_db       0
shimmer_apq3     0
shimmer_apq5     0
shimmer_apq11    0
shimmer_dda      0
nhr              0
hnr              0
rpde             0
dfa              0
ppe              0
dtype: int64

In [14]:
# Subject-level baseline (first observation per subject)
data["baseline_updrs"] = (
    data.groupby("subject")["motor_updrs"]
      .transform("first")
)

# Target becomes change from baseline
data["delta_motor_updrs"] = (
    data["motor_updrs"] - data["baseline_updrs"]
)

data[["subject", "test_time", "motor_updrs", "baseline_updrs", "delta_motor_updrs"]].head(10)


,subject,test_time,motor_updrs,baseline_updrs,delta_motor_updrs
0,1,5.6431,28.199,28.199,0.000
1,1,12.6660,28.447,28.199,0.248
2,1,19.6810,28.695,28.199,0.496
3,1,25.6470,28.905,28.199,0.706
4,1,33.6420,29.187,28.199,0.988
5,1,40.6520,29.435,28.199,1.236
6,1,47.6490,29.682,28.199,1.483
7,1,54.6400,29.928,28.199,1.729
8,1,61.6690,30.177,28.199,1.978
9,1,68.6880,30.424,28.199,2.225


In [16]:
# Lag features
data["delta_lag_1"] = (
    data.groupby("subject")["delta_motor_updrs"]
      .shift(1)
)

# Time difference
data["time_diff"] = (
    data.groupby("subject")["test_time"]
      .diff()
)

# Rolling slope (trend)
data["delta_trend_3"] = (
    data.groupby("subject")["delta_motor_updrs"]
      .rolling(window=3, min_periods=2)
      .mean()
      .reset_index(level=0, drop=True)
)

# Drop rows with NaNs from lags
data = data.dropna().reset_index(drop=True)


In [31]:
data.columns

Index(['subject', 'age', 'sex', 'test_time', 'motor_updrs', 'total_updrs',
       'jitter', 'jitter_abs', 'jitter_rap', 'jitter_ppq5', 'jitter_ddp',
       'shimmer', 'shimmer_db', 'shimmer_apq3', 'shimmer_apq5',
       'shimmer_apq11', 'shimmer_dda', 'nhr', 'hnr', 'rpde', 'dfa', 'ppe',
       'baseline_updrs', 'delta_motor_updrs', 'delta_lag_1', 'time_diff',
       'delta_trend_3', 'voice_instability_mean', 'voice_instability_std',
       'jitter_shimmer_ratio', 'log_ppe', 'log_hnr', 'dysphonia_index',
       'age_jitter_interaction', 'age_shimmer_interaction'],
      dtype='object')

In [18]:
data["voice_instability_mean"] = data[["jitter", "shimmer"]].mean(axis=1)
data["voice_instability_std"] = data[["jitter", "shimmer"]].std(axis=1)



In [20]:
data["jitter_shimmer_ratio"] = data["jitter"] / (data["shimmer"] + 1e-6)
data["log_ppe"] = np.log1p(data["ppe"])
data["log_hnr"] = np.log1p(data["hnr"])


In [23]:
data["dysphonia_index"] = (
    data["jitter"].rank(pct=True) +
    data["shimmer"].rank(pct=True) +
    data["ppe"].rank(pct=True) -
    data["hnr"].rank(pct=True)
)


In [25]:
data["age_jitter_interaction"] = data["age"] * data["jitter"]
data["age_shimmer_interaction"] = data["age"] * data["shimmer"]


In [49]:
static_features = [
    # Identifiers (kept for grouping, NOT modeling)
    "subject",
    "test_time",

    # Target
    "motor_updrs",

    # Core voice features
    "jitter", "shimmer", "hnr",
    "dfa", "rpde", "ppe",

    # Demographics
    "age", "sex",

    # Engineered features
    "voice_instability_mean",
    "voice_instability_std",
    "jitter_shimmer_ratio",
    "log_ppe",
    "log_hnr",
    "dysphonia_index",
    "age_jitter_interaction",
    "age_shimmer_interaction",
    "delta_lag_1",
    "time_diff",
    "delta_trend_3"
]


features_static = data[static_features]

In [45]:
final_columns = (
    ["subject", "test_time"] +
    static_features +
    ["baseline_updrs", "delta_motor_updrs"]
)

featured_df = data[final_columns].copy()
featured_df.head()


,subject,test_time,subject,test_time,motor_updrs,jitter,shimmer,hnr,dfa,rpde,...,voice_instability_mean,voice_instability_std,jitter_shimmer_ratio,log_ppe,log_hnr,dysphonia_index,age_jitter_interaction,age_shimmer_interaction,baseline_updrs,delta_motor_updrs
0,1,12.666,1,12.666,28.447,0.00300,0.02024,27.183,0.56477,0.43493,...,0.011620,0.012191,0.148214,0.102647,3.338719,-0.421653,0.21600,1.45728,28.199,0.248
1,1,19.681,1,19.681,28.695,0.00481,0.01675,23.047,0.54405,0.46222,...,0.010780,0.008443,0.287147,0.190736,3.180010,0.561632,0.34632,1.20600,28.199,0.496
2,1,25.647,1,25.647,28.905,0.00528,0.02309,24.445,0.57794,0.48730,...,0.014185,0.012594,0.228661,0.287259,3.236519,1.084348,0.38016,1.66248,28.199,0.706
3,1,33.642,1,33.642,29.187,0.00335,0.01703,26.126,0.56122,0.47188,...,0.010190,0.009673,0.196700,0.176982,3.300493,-0.045860,0.24120,1.22616,28.199,0.988
4,1,40.652,1,40.652,29.435,0.00353,0.02227,22.946,0.57243,0.53949,...,0.012900,0.013251,0.158502,0.178146,3.175801,0.436311,0.25416,1.60344,28.199,1.236


In [47]:
featured_df.isnull().sum()
# featured_df.describe()

subject                    0
test_time                  0
subject                    0
test_time                  0
motor_updrs                0
jitter                     0
shimmer                    0
hnr                        0
dfa                        0
rpde                       0
ppe                        0
age                        0
sex                        0
voice_instability_mean     0
voice_instability_std      0
jitter_shimmer_ratio       0
log_ppe                    0
log_hnr                    0
dysphonia_index            0
age_jitter_interaction     0
age_shimmer_interaction    0
baseline_updrs             0
delta_motor_updrs          0
dtype: int64

In [41]:
featured_df.to_csv("C:/Users/Durosimi/PROJECTS/Parkinson/data/processed/features_df.csv", index=False)

In [ ]:
# set(features_static.columns) - set(raw_data.columns)


SPLITTING THE DATA INTO TRAIN AND TEST DATA. SPLITTING BY PATIENTS, NOT BY ROWS

In [ ]:
# from sklearn.model_selection import GroupShuffleSplit

In [ ]:
# gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# train_idx, test_idx = next( gss.split(data, groups=data[id_column]))

# train_data = data.iloc[train_idx]
# test_data = data.iloc[test_idx]

In [ ]:
# # Check if any subject appears in both sets
# train_subjects = set(train_data[id_column])
# test_subjects = set(test_data[id_column])
# common = train_subjects.intersection(test_subjects)
# print(f"Subjects in both sets: {common}")  # Should be empty!

In [ ]:
# train_data["sex"] = train_data["sex"].astype(int)
# test_data["sex"]  = test_data["sex"].astype(int)

In [ ]:
# train_data.to_csv("C://Users//Durosimi//PROJECTS//Parkinson/train_data.csv")

In [ ]:
# test_data.to_csv("C://Users//Durosimi//PROJECTS//Parkinson/test_data.csv")

In [ ]:
# from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# selected_features = [
#     "jitter",
#     "shimmer",
#     "dfa",
#     "age",
#     "sex"
# ]

In [ ]:
# X = train_data[selected_features]

In [ ]:
# X.dtypes

In [ ]:
# X = X.apply(pd.to_numeric, errors="coerce")

In [ ]:
# from statsmodels.stats.outliers_influence import variance_inflation_factor
# import numpy as np
# import pandas as pd

# # Ensure all features are numeric floats
# X = X.apply(pd.to_numeric, errors="coerce")
# X = X.replace([np.inf, -np.inf], np.nan)
# X = X.dropna()

# X_values = X.values.astype(float)

# vif = pd.DataFrame()
# vif["feature"] = X.columns
# vif["VIF"] = [
#     variance_inflation_factor(X_values, i)
#     for i in range(X_values.shape[1])
# ]

# vif


In [ ]:
# from sklearn.preprocessing import StandardScaler
# import joblib

In [ ]:
#SCALING

In [ ]:
# scaler_linear = StandardScaler()

# train_data[selected_features] = scaler_linear.fit_transform(
#     train_data[selected_features]
# )

# test_data[selected_features] = scaler_linear.transform(
#     test_data[selected_features]
# )

# joblib.dump(
#     scaler_linear,
#     "C:/Users/Durosimi/PROJECTS/Parkinson/models_artifacts/scaler_linear.pkl"
# )

In [ ]:
#SAVE LINEAR FEATURES

In [ ]:
# linear_cols = selected_features + target

# train_data[linear_cols].to_csv(
#     "C:/Users/Durosimi/PROJECTS/Parkinson/data/features/train_linear.csv",
#     index=False
# )

# test_data[linear_cols].to_csv(
#     "C:/Users/Durosimi/PROJECTS/Parkinson/data/features/test_linear.csv",
#     index=False
# )

In [ ]:
# tree_features = [
#     "jitter",
#     "jitter_ddp",
#     "shimmer",
#     "shimmer_dda",
#     "hnr",
#     "rpde",
#     "dfa",
#     "ppe",
#     "age",
#     "sex"
# ]

In [ ]:
# scaler_tree = StandardScaler()

# train_data[tree_features] = scaler_tree.fit_transform(
#     train_data[tree_features]
# )

# test_data[tree_features] = scaler_tree.transform(
#     test_data[tree_features]
# )

# joblib.dump(
#     scaler_tree,
#     "C:/Users/Durosimi/PROJECTS/Parkinson/models_artifacts/scaler_tree.pkl"
# )

In [ ]:
# tree_cols = tree_features + target

# train_data[tree_cols].to_csv(
#     "C:/Users/Durosimi/PROJECTS/Parkinson/data/features/train_tree.csv",
#     index=False
# )

# test_data[tree_cols].to_csv(
#     "C:/Users/Durosimi/PROJECTS/Parkinson/data/features/test_tree.csv",
#     index=False
# )

In [ ]:
# import os

# OUTPUT_PATH = "C:/Users/Durosimi/PROJECTS/Parkinson/data/processed"
# os.makedirs(OUTPUT_PATH, exist_ok=True)

# data.to_csv(
#     os.path.join(OUTPUT_PATH, "featured_data.csv"),
#     index=False
# )

# print("✅ featured_data.csv saved to data/processed/")


In [ ]:
# df = pd.read_csv("C:/Users/Durosimi/PROJECTS/Parkinson/data/processed/featured_data.csv")

In [ ]:
# df.head()

In [ ]:
# data["jitter_shimmer_ratio"] = data["jitter"] / (data["shimmer"] + 1e-6)
# data["log_ppe"] = np.log1p(data["ppe"])
# data["voice_instability"] = data[["jitter", "shimmer"]].mean(axis=1)


In [ ]:
features_static.to_csv("C:/Users/Durosimi/PROJECTS/Parkinson/data/processed/features_static.csv", index=False)


In [ ]:
features_static.shape
# features_static.head()

In [ ]:
raw_data.shape, features_static.shape


### Feature Engineering Summary

- Patient-level group split applied to avoid leakage
- Severe multicollinearity detected in EDA
- Linear models use pruned, interpretable feature set
- Tree-based models retain full acoustic feature space
- Features standardized using StandardScaler
- Separate datasets saved for each modeling family

This setup supports robust benchmarking, interpretability,
and production deployment.